# Titanic survival prediction (Lab 5)

Predict whether each passenger in the test set survived (`1`) or not (`0`). The score is accuracy.

The gender-only file `gender_submission.csv` scores **0.74850** on the public leaderboard. This notebook builds a model from `train.csv` only. It does not use any external solution file or test labels.

`RANDOM_STATE = 42` is fixed so the same code produces the same submission.

Run the sections in order. Each section stops at a result you can read before the next one changes the approach.

## 1. Look at the data

Load the training and test files, check what is missing, and measure the gender rule on the training set. No new features and no model in this section.

In [1]:
RANDOM_STATE = 42

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

np.random.seed(RANDOM_STATE)
pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 120)

In [2]:
train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")

print("train", train.shape)
print("test", test.shape)
train.head()

train (891, 12)
test (418, 11)


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [3]:
print("Training dtypes")
print(train.dtypes)
print()
print("Missing values")
missing = pd.DataFrame({
    "train": train.drop(columns="Survived").isna().sum(),
    "test": test.isna().sum(),
})
missing

Training dtypes
PassengerId      int64
Survived         int64
Pclass           int64
Name            object
Sex             object
Age            float64
SibSp            int64
Parch            int64
Ticket          object
Fare           float64
Cabin           object
Embarked        object
dtype: object

Missing values


,train,test
PassengerId,0,0
Pclass,0,0
Name,0,0
Sex,0,0
Age,177,86
SibSp,0,0
Parch,0,0
Ticket,0,0
Fare,0,1
Cabin,687,327


In [4]:
print(f"Overall survival rate: {train['Survived'].mean():.3f} ({train['Survived'].sum()} of {len(train)})")
print()
print("Survival rate by Sex")
print(train.groupby("Sex")["Survived"].agg(["mean", "count"]))
print()
print("Survival rate by Pclass")
print(train.groupby("Pclass")["Survived"].agg(["mean", "count"]))
print()
print("Survival rate by Sex and Pclass")
print(train.pivot_table(index="Sex", columns="Pclass", values="Survived", aggfunc="mean").round(3))

Overall survival rate: 0.384 (342 of 891)

Survival rate by Sex
            mean  count
Sex                    
female  0.742038    314
male    0.188908    577

Survival rate by Pclass
            mean  count
Pclass                 
1       0.629630    216
2       0.472826    184
3       0.242363    491

Survival rate by Sex and Pclass
Pclass      1      2      3
Sex                        
female  0.968  0.921  0.500
male    0.369  0.157  0.135


In [5]:
fig, axes = plt.subplots(1, 2, figsize=(8, 3.5))

train.groupby("Sex")["Survived"].mean().plot(kind="bar", ax=axes[0], color=["C1", "C0"], rot=0)
axes[0].set_ylim(0, 1)
axes[0].set_ylabel("Survival rate")
axes[0].set_title("By sex")

train.groupby("Pclass")["Survived"].mean().plot(kind="bar", ax=axes[1], color="C2", rot=0)
axes[1].set_ylim(0, 1)
axes[1].set_ylabel("Survival rate")
axes[1].set_title("By passenger class")

fig.tight_layout()
fig

<Figure size 800x350 with 2 Axes>

In [6]:
PUBLIC_BASELINE = 0.74850

gender_pred = (train["Sex"] == "female").astype(int)
gender_accuracy = (gender_pred == train["Survived"]).mean()

print(f"Gender rule on the training set: predict 1 if female, else 0")
print(f"Training accuracy: {gender_accuracy:.5f}")
print(f"Public leaderboard score of gender_submission.csv: {PUBLIC_BASELINE:.5f}")
print()
print(
    "Training accuracy is higher than the public score because they are measured on different passengers. "
    "The checkpoint still requires a public score above 0.74850."
)

Gender rule on the training set: predict 1 if female, else 0
Training accuracy: 0.78676
Public leaderboard score of gender_submission.csv: 0.74850

Training accuracy is higher than the public score because they are measured on different passengers. The checkpoint still requires a public score above 0.74850.


## 2. Features

Add one feature at a time and check the training survival rate before using it in a model. Filled-in values use statistics from the training set only, then the same numbers are applied to the test set.

The gender rule treats every man the same. Title separates boys (`Master`) from adult men (`Mr`), and family size separates people traveling alone from large groups.

In [7]:
def title_from_name(name):
    """Bucket the honorific in a passenger name.

    Mme and Ms are treated as Mrs. Mlle is treated as Miss.
    Anything other than Mr, Mrs, Miss, or Master is Rare.
    """
    title = name.split(",")[1].split(".")[0].strip()
    if title in {"Mme", "Ms"}:
        return "Mrs"
    if title == "Mlle":
        return "Miss"
    if title in {"Mr", "Mrs", "Miss", "Master"}:
        return title
    return "Rare"


train["Title"] = train["Name"].map(title_from_name)
test["Title"] = test["Name"].map(title_from_name)

print("Title counts and survival rate (training)")
print(train.groupby("Title")["Survived"].agg(["mean", "count"]).sort_values("mean", ascending=False))

Title counts and survival rate (training)
            mean  count
Title                  
Mrs     0.795276    127
Miss    0.701087    184
Master  0.575000     40
Rare    0.347826     23
Mr      0.156673    517


In [8]:
for frame in (train, test):
    frame["FamilySize"] = frame["SibSp"] + frame["Parch"] + 1
    frame["IsAlone"] = (frame["FamilySize"] == 1).astype(int)

print("Survival rate by family size (training)")
print(train.groupby("FamilySize")["Survived"].agg(["mean", "count"]))
print()
print("Survival rate by traveling alone")
print(train.groupby("IsAlone")["Survived"].agg(["mean", "count"]))

Survival rate by family size (training)
                mean  count
FamilySize                 
1           0.303538    537
2           0.552795    161
3           0.578431    102
4           0.724138     29
5           0.200000     15
6           0.136364     22
7           0.333333     12
8           0.000000      6
11          0.000000      7

Survival rate by traveling alone
             mean  count
IsAlone                 
0        0.505650    354
1        0.303538    537


In [9]:
age_by_title = train.groupby("Title")["Age"].median()
age_global = train["Age"].median()

print("Median age by title, from the training set")
print(age_by_title)
print(f"Overall training median age: {age_global}")

for frame in (train, test):
    frame["Age"] = frame["Age"].fillna(frame["Title"].map(age_by_title))
    frame["Age"] = frame["Age"].fillna(age_global)

print()
print("Missing Age after fill:", int(train["Age"].isna().sum()), "train,", int(test["Age"].isna().sum()), "test")

Median age by title, from the training set
Title
Master     3.5
Miss      21.0
Mr        30.0
Mrs       35.0
Rare      48.5
Name: Age, dtype: float64
Overall training median age: 28.0

Missing Age after fill: 0 train, 0 test


In [10]:
fare_median = train["Fare"].median()
print(f"Training median fare used to fill gaps: {fare_median}")

for frame in (train, test):
    frame["Fare"] = frame["Fare"].fillna(fare_median)
    frame["LogFare"] = np.log1p(frame["Fare"])

train["FareBand"] = pd.qcut(train["Fare"], 4, duplicates="drop")
print()
print("Survival rate by fare quartile (training)")
print(train.groupby("FareBand", observed=True)["Survived"].agg(["mean", "count"]))
train = train.drop(columns="FareBand")

Training median fare used to fill gaps: 14.4542

Survival rate by fare quartile (training)
                     mean  count
FareBand                        
(-0.001, 7.91]   0.197309    223
(7.91, 14.454]   0.303571    224
(14.454, 31.0]   0.454955    222
(31.0, 512.329]  0.581081    222


In [11]:
embarked_mode = train["Embarked"].mode().iloc[0]
print(f"Most common embarkation port in training: {embarked_mode}")
print()
print("Survival rate by Embarked, before filling the 2 missing training values")
print(train.groupby("Embarked", dropna=False)["Survived"].agg(["mean", "count"]))

for frame in (train, test):
    frame["Embarked"] = frame["Embarked"].fillna(embarked_mode)

print()
print("Missing Embarked after fill:", int(train["Embarked"].isna().sum()), "train,", int(test["Embarked"].isna().sum()), "test")

Most common embarkation port in training: S

Survival rate by Embarked, before filling the 2 missing training values
              mean  count
Embarked                 
C         0.553571    168
Q         0.389610     77
S         0.336957    644
NaN       1.000000      2

Missing Embarked after fill: 0 train, 0 test


In [12]:
for frame in (train, test):
    frame["HasCabin"] = frame["Cabin"].notna().astype(int)

print("Survival rate by whether a cabin was recorded (training)")
print(train.groupby("HasCabin")["Survived"].agg(["mean", "count"]))
print()
print("Cabin is missing for most passengers, so the model uses this yes/no flag instead of the cabin string.")

Survival rate by whether a cabin was recorded (training)
              mean  count
HasCabin                 
0         0.299854    687
1         0.666667    204

Cabin is missing for most passengers, so the model uses this yes/no flag instead of the cabin string.


In [13]:
model_columns = [
    "Pclass", "Sex", "Title", "Age", "LogFare", "FamilySize", "IsAlone", "Embarked", "HasCabin",
]
print("Training rows after feature steps")
print("Missing values in model columns:", int(train[model_columns].isna().sum().sum()))
train[["Survived", *model_columns]].head(8)

Training rows after feature steps
Missing values in model columns: 0


,Survived,Pclass,Sex,Title,Age,LogFare,FamilySize,IsAlone,Embarked,HasCabin
0,0,3,male,Mr,22.0,2.110213,2,0,S,0
1,1,1,female,Mrs,38.0,4.280593,2,0,C,1
2,1,3,female,Miss,26.0,2.188856,1,1,S,0
3,1,1,female,Mrs,35.0,3.990834,2,0,S,1
4,0,3,male,Mr,35.0,2.202765,1,1,S,0
5,0,3,male,Mr,30.0,2.246893,1,1,Q,0
6,0,1,male,Mr,54.0,3.967694,1,1,S,1
7,0,3,male,Master,2.0,3.094446,5,0,S,0


## 3. Models and submission

Compare three classifiers with the same features. Scores are accuracy from stratified 5-fold cross-validation on the training set, with `random_state=42`. The gender rule is scored on the same folds.

The model with the highest mean fold accuracy is fit on all 891 training rows and applied to the test set. The decision threshold stays at 0.5.

In [14]:
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

categorical_columns = ["Pclass", "Sex", "Title", "Embarked"]
numeric_columns = ["Age", "LogFare", "FamilySize", "IsAlone", "HasCabin"]

preprocess = ColumnTransformer(
    transformers=[
        (
            "categorical",
            OneHotEncoder(handle_unknown="ignore", sparse_output=False),
            categorical_columns,
        ),
        ("numeric", StandardScaler(), numeric_columns),
    ]
)

models = {
    "logistic_regression": LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
    "random_forest": RandomForestClassifier(
        n_estimators=300,
        max_depth=6,
        min_samples_leaf=3,
        random_state=RANDOM_STATE,
    ),
    "hist_gradient_boosting": HistGradientBoostingClassifier(
        max_depth=3,
        learning_rate=0.1,
        max_iter=200,
        random_state=RANDOM_STATE,
    ),
}

In [15]:
import warnings

# This NumPy build warns inside logistic regression's matrix multiply.
# The warning does not change the accuracy scores below.
warnings.filterwarnings("ignore", message=".*matmul.*", category=RuntimeWarning)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
y = train["Survived"]
X = train[categorical_columns + numeric_columns]

gender_fold_scores = []
for _, valid_idx in cv.split(X, y):
    fold_pred = (train.iloc[valid_idx]["Sex"] == "female").astype(int)
    gender_fold_scores.append((fold_pred.to_numpy() == y.iloc[valid_idx].to_numpy()).mean())

rows = [{
    "model": "gender_rule",
    "mean_accuracy": float(np.mean(gender_fold_scores)),
    "std_accuracy": float(np.std(gender_fold_scores)),
}]

fold_scores = {}
for name, estimator in models.items():
    pipeline = Pipeline([("preprocess", preprocess), ("model", estimator)])
    scores = []
    for train_idx, valid_idx in cv.split(X, y):
        pipeline.fit(X.iloc[train_idx], y.iloc[train_idx])
        pred = pipeline.predict(X.iloc[valid_idx])
        scores.append((pred == y.iloc[valid_idx].to_numpy()).mean())
    fold_scores[name] = scores
    rows.append({
        "model": name,
        "mean_accuracy": float(np.mean(scores)),
        "std_accuracy": float(np.std(scores)),
    })

comparison = pd.DataFrame(rows).sort_values("mean_accuracy", ascending=False).reset_index(drop=True)
comparison

,model,mean_accuracy,std_accuracy
0,hist_gradient_boosting,0.840613,0.021008
1,random_forest,0.831630,0.013512
2,logistic_regression,0.829383,0.013817
3,gender_rule,0.786755,0.018807


In [16]:
best_name = comparison.loc[comparison["model"] != "gender_rule", "model"].iloc[0]
print(f"Selected model: {best_name}")

best_pipeline = Pipeline([
    ("preprocess", preprocess),
    ("model", models[best_name]),
])
best_pipeline.fit(X, y)

test_pred = best_pipeline.predict(test[categorical_columns + numeric_columns]).astype(int)
submission = pd.DataFrame({
    "PassengerId": test["PassengerId"],
    "Survived": test_pred,
})
submission.to_csv("submission.csv", index=False)
print(f"Wrote submission.csv with {len(submission)} rows")
print(f"Predicted survival rate: {submission['Survived'].mean():.3f}")
submission.head()

Selected model: hist_gradient_boosting


Wrote submission.csv with 418 rows
Predicted survival rate: 0.361


,PassengerId,Survived
0,892,0
1,893,0
2,894,0
3,895,0
4,896,1


In [17]:
check = pd.read_csv("submission.csv")
assert list(check.columns) == ["PassengerId", "Survived"]
assert len(check) == 418
assert check["PassengerId"].tolist() == test["PassengerId"].tolist()
assert set(check["Survived"].unique()) <= {0, 1}
assert check["Survived"].isna().sum() == 0
print("submission.csv has 418 rows, matching PassengerIds, and binary Survived values.")

submission.csv has 418 rows, matching PassengerIds, and binary Survived values.


## Next step

Upload `submission.csv` to the Kaggle competition. It has 418 rows and the columns `PassengerId` and `Survived`. The checkpoint is a public score above **0.74850**. Kaggle allows 5 submissions per day, resetting at 5:00pm PT.

On Gradescope, submit a screenshot of the team name, rank, and accuracy for Lab 5a.